In [ ]:
# =====================================================================
# SYSTEM COMPONENT: IMPORTATION DES LIBRAIRIES ET CONFIGURATION GLOBAL
# =====================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Fixer les seeds pour la reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# --- CONFIGURATION VARIABLES GLOBALES ---
# Note : Réduction à 48x48 imposée par l'énoncé en raison de la taille des images originales
IMG_HEIGHT = 48
IMG_WIDTH = 48
BATCH_SIZE = 32
CHANNELS = 3
EPOCHS = 15  # Ajustable selon les ressources

print("✅ Environnement TensorFlow initialisé avec succès.")

# =====================================================================
# NOTE ESSENTIELLE SUR LE JEU DE DONNÉES (Dossiers Google Colab)
# =====================================================================
# Avant d'exécuter la cellule suivante, assurez-vous d'avoir téléchargé
# et extrait votre dataset dans l'environnement Colab.
# Exemple de structure attendue :
# data_dir/
#    train/
#       Astilbe/ ...
#       Iris/ ...
#    validation/
#       Astilbe/ ...

# Pour l'exercice, nous définissons des chemins génériques à adapter si besoin :
DATA_DIR = './flower_dataset' 
TRAIN_DIR = os.path.join(DATA_DIR, 'train') if os.path.exists(os.path.join(DATA_DIR, 'train')) else DATA_DIR
VAL_DIR = os.path.join(DATA_DIR, 'validation') if os.path.exists(os.path.join(DATA_DIR, 'validation')) else DATA_DIR


# =====================================================================
# PART 1: DATA EXPLORATION AND VISUALIZATION
# =====================================================================
print("\n--- PART 1: CHARGEMENT ET EXPLORATION DES DONNÉES ---")

# Chargement sécurisé via tf.keras.utils.image_dataset_from_directory
try:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        label_mode='int'
    )
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        label_mode='int'
    )
    
    class_names = train_ds.class_names
    print(f"\n✅ Classes détectées ({len(class_names)}) : {class_names}")
    
except Exception as e:
    print(f"⚠️ Erreur de chargement. Veuillez vérifier vos dossiers. Simulation de données en cours pour validation du code.")
    # Fallback sécurisé en cas d'absence des fichiers locaux lors de l'exécution initiale
    class_names = ['Astilbe', 'Bellflower', 'Black-eyed Susan', 'Calendula', 'California Poppy', 
                   'Carnation', 'Common Daisy', 'Coreopsis', 'Dandelion', 'Iris', 'Rose', 'Sunflower', 'Tulip', 'Water Lily']
    train_ds = tf.data.Dataset.from_tensor_slices((np.random.rand(100, IMG_HEIGHT, IMG_WIDTH, CHANNELS), np.random.randint(0, 14, 100))).batch(BATCH_SIZE)
    val_ds = tf.data.Dataset.from_tensor_slices((np.random.rand(20, IMG_HEIGHT, IMG_WIDTH, CHANNELS), np.random.randint(0, 14, 20))).batch(BATCH_SIZE)

# --- Fonction de Visualisation Grille 3x3 ---
def visualize_images_grid(dataset, class_names):
    plt.figure(figsize=(10, 10))
    for images, labels in dataset.take(1):
        for i in range(9):
            if i < len(images):
                ax = plt.subplot(3, 3, i + 1)
                plt.imshow(images[i].numpy().astype("uint8"))
                plt.title(class_names[labels[i]])
                plt.axis("off")
    plt.suptitle("Grille d'exploration 3x3 - Aperçu des Images", fontsize=16)
    plt.show()

# Exécution de la visualisation
visualize_images_grid(train_ds, class_names)

"""
--- Analyse des défis de classification (Réponses théoriques Partie 1) ---
1. Perte de résolution : Passer à une dimension de 48x48 pixels élimine une quantité considérable de détails fins (textures des pétales, étamines). Le modèle doit s'appuyer sur des motifs structurels globaux et de grands blocs de couleurs.
2. Similarité morphologique : Certaines espèces de la famille des astéracées (Common Daisy, Dandelion, Sunflower, Coreopsis) possèdent des formes géométriques et des teintes très proches (jaune/blanc), induisant de potentiels risques de confusion.
3. Variabilité intra-classe : Les fleurs comme les Roses ou les Tulipes se déclinent sous de multiples variétés chromatiques (rouge, blanc, jaune), rendant l'association de couleurs brutes parfois trompeuse pour un réseau non optimisé.
"""


# =====================================================================
# PART 4: DATA AUGMENTATION (Placée en amont de l'architecture)
# =====================================================================
# L'utilisation d'une couche d'augmentation native au sein du modèle est plus performante
# qu'un ImageDataGenerator classique car elle s'exécute directement sur GPU.
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1)
], name="Data_Augmentation_Layer")


# =====================================================================
# PART 2 & 3: MODEL ARCHITECTURE DESIGN & HYPERPARAMETER TUNING
# =====================================================================
print("\n--- PART 2 & 3: CONSTRUCTION ET COMPILATION DU MODELE CNN ---")

def build_robust_cnn(input_shape, num_classes):
    model = models.Sequential()
    
    # Entrée du modèle
    model.add(layers.Input(shape=input_shape))
    
    # Application de l'augmentation des données à l'entraînement uniquement
    model.add(data_augmentation)
    
    # Normalisation des pixels [0, 1]
    model.add(layers.Rescaling(1./255))
    
    # Premier bloc Convolutionnel
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    # Deuxième bloc Convolutionnel (Extraction de caractéristiques moyennes)
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    # Troisième bloc Convolutionnel (Motifs complexes)
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    
    # Couches denses de Classification
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5)) # Régularisation stricte pour contrer l'overfitting
    
    # Couche de sortie (14 classes de fleurs)
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    return model

# Initialisation du modèle
model = build_robust_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), num_classes=len(class_names))
model.summary()

# --- Optimisation et Callbacks (Part 3) ---
# Utilisation de l'optimiseur Adam avec un taux d'apprentissage stable de 0.001
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks stratégiques pour automatiser la convergence et prévenir le surapprentissage
callbacks_list = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, verbose=1)
]

# --- Entraînement du modèle ---
print("\n Lancement de la phase d'entraînement...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    verbose=1
)


# =====================================================================
# PART 5: PERFORMANCE EVALUATION AND ANALYSIS
# =====================================================================
print("\n--- PART 5: EVALUATION DES PERFORMANCES ---")

# 1. Courbes d'apprentissage (Précision et Perte)
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))

# Graphique d'exactitude
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Accuracy Entraînement', color='blue', lw=2)
plt.plot(epochs_range, val_acc, label='Accuracy Validation', color='orange', lw=2)
plt.legend(loc='lower right')
plt.title('Courbes d\'Exactitude (Accuracy Train vs Val)')
plt.xlabel('Epochs')
plt.ylabel('Score')

# Graphique de perte
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Loss Entraînement', color='blue', lw=2)
plt.plot(epochs_range, val_loss, label='Loss Validation', color='orange', lw=2)
plt.legend(loc='upper right')
plt.title('Courbes de Perte (Loss Train vs Val)')
plt.xlabel('Epochs')
plt.ylabel('Valeur Loss')

plt.tight_layout()
plt.show()

# 2. Rapport Métriques de Classification & Matrice de Confusion
print("\n Génération du rapport complet d'évaluation globale...")

all_images = []
all_labels = []

# Extraction des vrais labels du dataset de validation
for images, labels in val_ds:
    all_images.append(images.numpy())
    all_labels.append(labels.numpy())

val_images_concat = np.concatenate(all_images, axis=0)
val_labels_true = np.concatenate(all_labels, axis=0)

# Prédiction
predictions = model.predict(val_images_concat, verbose=0)
val_labels_pred = np.argmax(predictions, axis=1)

# Affichage du rapport de classification
print("\n📊 Classification Report :")
print(classification_report(val_labels_true, val_labels_pred, target_names=class_names, zero_division=0))

# Génération graphique de la Matrice de Confusion
cm = confusion_matrix(val_labels_true, val_labels_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Matrice de Confusion Finale')
plt.ylabel('Vraies Classes')
plt.xlabel('Classes Prédites')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


# =====================================================================
# PART 6: MODEL SAVING
# =====================================================================
print("\n--- PART 6: ENREGISTREMENT DU MODELE ---")
model_save_path = 'flower_classification_cnn_model.h5'
model.save(model_save_path)
print(f"✅ Modèle sauvegardé avec succès à l'emplacement : '{model_save_path}'")